# [Open in Colab](https://colab.research.google.com/github/dmochow/BME-i9400-2026/blob/main/capstone/starter_C_ecg.ipynb)

# BME i9400 — Capstone starter C
## Heartbeat Classification in Patients the Model Has Not Seen

This notebook downloads the recordings, extracts the beats, applies the fixed split, and fits the
baseline. Run it from top to bottom before you change anything; it takes about two minutes.

---
## 1 — The data

The MIT-BIH Arrhythmia Database (Moody and Mark, "The impact of the MIT-BIH Arrhythmia Database",
*IEEE Engineering in Medicine and Biology Magazine*, 2001), distributed by PhysioNet under the Open
Data Commons Attribution License: 48 half-hour, two-lead ambulatory ECG recordings from 47 patients,
sampled at 360 Hz, in which two cardiologists independently annotated every beat.

- **A row of $X$**: one heartbeat. A window of the lead II (MLII) signal from 0.25 s before to 0.44 s
  after the annotated R peak (250 samples), and six RR-interval features.
- **$y$**: the beat class of the ANSI/AAMI EC57 standard:

| Class | Name | Annotations included |
|---|---|---|
| N | normal and bundle branch block beats | `N`, `L`, `R`, `e`, `j` |
| S | supraventricular ectopic beats | `A`, `a`, `J`, `S` |
| V | ventricular ectopic beats | `V`, `E` |
| F | fusion of ventricular and normal beats | `F` |

> The **RR interval** is the time between two consecutive R peaks, the inverse of the instantaneous
> heart rate. An ectopic beat usually arrives early (a short preceding RR interval) and is followed
> by a pause (a long following one).
>
> The fifth AAMI class, Q (unclassifiable), has 15 beats in these recordings and is excluded, as in
> most published work.

In [ ]:
!pip install -q wfdb

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import wfdb
from scipy.signal import butter, filtfilt
plt.rcParams.update({"font.size": 12, "figure.dpi": 110})

# The inter-patient split of de Chazal et al. (2004). The four records with paced beats (102, 104, 107, 217) are excluded.
DS1 = [101, 106, 108, 109, 112, 114, 115, 116, 118, 119, 122, 124, 201, 203, 205, 207, 208, 209, 215, 220, 223, 230]
DS2 = [100, 103, 105, 111, 113, 117, 121, 123, 200, 202, 210, 212, 213, 214, 219, 221, 222, 228, 231, 232, 233, 234]
CLASSES = ["N", "S", "V", "F"]
AAMI = {**{s: 0 for s in "NLRej"}, **{s: 1 for s in "aAJS"}, **{s: 2 for s in "VE"}, "F": 3}

import contextlib, io
with contextlib.redirect_stdout(io.StringIO()):          # wfdb prints one line per file
    wfdb.dl_database("mitdb", "mitdb", records=[str(r) for r in DS1 + DS2], annotators=["atr"])
print("downloaded", len(DS1 + DS2), "records")

---
## 2 — Extracting the beats

Each recording is band-pass filtered (0.5–40 Hz) to remove baseline wander and high-frequency noise.
For each annotated beat of classes N, S, V, F (except the first and last of a recording), the window
around the R peak and six RR features are stored:

- the preceding and following RR intervals, in seconds
- the mean of the ten preceding RR intervals (the local rhythm)
- the preceding and following RR intervals divided by the recording's mean RR interval, and the
  preceding interval divided by the local mean

> Dividing by the recording's own mean RR interval adjusts for differences in resting heart rate
> between patients. It uses only the unlabelled signal of that recording, which is also available
> when the model is applied to a new patient.

In [ ]:
FS, PRE, POST = 360, 90, 160
b, a = butter(2, [0.5, 40], btype="band", fs=FS)

def extract(records):
    waves, rr_feats, labels, recs = [], [], [], []
    for r in records:
        rec, ann = wfdb.rdrecord(f"mitdb/{r}"), wfdb.rdann(f"mitdb/{r}", "atr")
        x = filtfilt(b, a, rec.p_signal[:, rec.sig_name.index("MLII")])
        beats = [(s, sym) for s, sym in zip(ann.sample, ann.symbol) if sym in AAMI or sym in "/fQ"]
        peaks = np.array([s for s, _ in beats]); rr = np.diff(peaks) / FS; mean_rr = rr.mean()
        for i in range(1, len(beats) - 1):
            sym = beats[i][1]
            if sym not in AAMI or peaks[i] - PRE < 0 or peaks[i] + POST > len(x):
                continue
            local = rr[max(0, i - 10):i].mean()
            waves.append(x[peaks[i] - PRE:peaks[i] + POST])
            rr_feats.append([rr[i - 1], rr[i], local, rr[i - 1] / mean_rr, rr[i] / mean_rr, rr[i - 1] / local])
            labels.append(AAMI[sym]); recs.append(r)
    return (np.array(waves, np.float32), np.array(rr_feats, np.float32), np.array(labels), np.array(recs))

W_dev, RR_dev, y_dev, rec_dev = extract(DS1)
W_test, RR_test, y_test, rec_test = extract(DS2)
print("development (DS1):", dict(zip(CLASSES, np.bincount(y_dev, minlength=4).tolist())), f"  {len(DS1)} records")
print("test        (DS2):", dict(zip(CLASSES, np.bincount(y_test, minlength=4).tolist())), f"  {len(DS2)} records")

t = np.arange(-PRE, POST) / FS * 1000
fig, axes = plt.subplots(1, 4, figsize=(14, 3), sharey=True)
for k, ax in enumerate(axes):
    ix = np.flatnonzero(y_dev == k)[:200]
    ax.plot(t, W_dev[ix].T, color="C0", alpha=0.05); ax.plot(t, W_dev[ix].mean(0), color="k")
    ax.set_title(f"{CLASSES[k]} (first 200 beats)"); ax.set_xlabel("ms from R peak")
axes[0].set_ylabel("mV"); plt.tight_layout(); plt.show()

---
## 3 — The fixed split, and validation by record

The **development set** is DS1 and the **test set** is DS2: 22 recordings each, from different
patients (de Chazal, O'Dwyer and Reilly, *IEEE Transactions on Biomedical Engineering*, 2004).

There is no fixed validation set. Validate by **grouped cross-validation** on DS1, with every beat of
a recording in the same fold (`GroupKFold`, `groups=rec_dev`). Some classes are concentrated in a few
recordings — most of DS1's S beats come from one record — so validation estimates are noisy, and the
spread across folds should be reported.

> A random split of beats, which many published papers use, places beats of the same patient in the
> training and test sets and gives accuracies near 99%. Under the inter-patient split the S class is
> the hardest; its sensitivity is the number to watch.

In [ ]:
from sklearn.model_selection import GroupKFold
print("S beats per DS1 record:", pd.Series(rec_dev[y_dev == 1]).value_counts().head(6).to_dict())
folds = GroupKFold(n_splits=5)

---
## 4 — The baseline

Multinomial logistic regression on the RR features and the beat window down-sampled by 5 (50
samples), standardised.

> The **primary metric** is the macro-averaged F1 score over the four classes: the F1 score of each
> class, with equal weight. Also report the sensitivity of each class.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import f1_score, recall_score, confusion_matrix

def baseline_features(W, RR):
    return np.hstack([W[:, ::5], RR])

baseline = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000))
pred_cv = cross_val_predict(baseline, baseline_features(W_dev, RR_dev), y_dev, cv=folds, groups=rec_dev)
fold_f1 = [f1_score(y_dev[te], pred_cv[te], average="macro") for _, te in folds.split(W_dev, y_dev, rec_dev)]

print(f"baseline, grouped CV on DS1: macro F1 {f1_score(y_dev, pred_cv, average='macro'):.3f} "
      f"(per fold {np.round(fold_f1, 2)})")
print("sensitivity per class:", dict(zip(CLASSES, np.round(recall_score(y_dev, pred_cv, average=None), 3).tolist())))
print(pd.DataFrame(confusion_matrix(y_dev, pred_cv), index=CLASSES, columns=CLASSES))
baseline.fit(baseline_features(W_dev, RR_dev), y_dev)

---
## 5 — The test set and the bootstrap

> **The test set is used once.** The cell below defines the test-set evaluation and the bootstrap
> confidence interval but does not run them. Run them at the end of the project, for your final model
> and for the baseline only, after every other decision has been made on validation data.

In [ ]:
def bootstrap_difference(metric, y, pred_new, pred_base, groups=None, n_boot=2000, seed=0):
    """95% bootstrap confidence interval for metric(y, pred_new) - metric(y, pred_base).

    Resamples whole groups (patients, records, abstracts) with replacement when `groups` is given,
    otherwise single examples. Both models are scored on the same resample, so the interval is for
    the paired difference (Meeting 20).
    """
    rng = np.random.default_rng(seed)
    y, pred_new, pred_base = np.asarray(y), np.asarray(pred_new), np.asarray(pred_base)
    if groups is None:
        groups = np.arange(len(y))
    uniq, inv = np.unique(np.asarray(groups), return_inverse=True)
    members = [np.flatnonzero(inv == k) for k in range(len(uniq))]
    diffs = []
    for _ in range(n_boot):
        idx = np.concatenate([members[k] for k in rng.integers(0, len(uniq), len(uniq))])
        diffs.append(metric(y[idx], pred_new[idx]) - metric(y[idx], pred_base[idx]))
    point = metric(y, pred_new) - metric(y, pred_base)
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    return point, lo, hi


def evaluate_on_test(pred_test_new, pred_test_base):
    """Macro F1 and per-class sensitivity on DS2, and the 95% CI for the difference, resampling records."""
    macro_f1 = lambda yt, yp: f1_score(yt, yp, average="macro", labels=[0, 1, 2, 3])
    for name, p in [("final model", pred_test_new), ("baseline", pred_test_base)]:
        sens = recall_score(y_test, p, average=None, labels=[0, 1, 2, 3])
        print(f"{name:12s} macro F1 {macro_f1(y_test, p):.3f}   sensitivity", dict(zip(CLASSES, np.round(sens, 3).tolist())))
    d, lo, hi = bootstrap_difference(macro_f1, y_test, pred_test_new, pred_test_base, groups=rec_test)
    print(f"difference in macro F1 {d:+.3f}, 95% CI [{lo:+.3f}, {hi:+.3f}]")

# At the end of the project, and only then:
# evaluate_on_test(final_model_predictions_on_DS2, baseline.predict(baseline_features(W_test, RR_test)))

---
## 6 — From here

The requirements, deliverables, and grading are in the capstone brief,
[`capstone/README.md`](https://github.com/dmochow/BME-i9400-2026/blob/main/capstone/README.md). In
short: reproduce the baseline above; search at least two further model families with every choice
made on validation data; evaluate the final model and the baseline once on the test set, with a 95%
bootstrap confidence interval for the difference; analyse the errors; write the model card.

Make a copy of this notebook (`File ▸ Save a copy in Drive`) and build your study in it.

Directions specific to this dataset: a 1D convolutional network on the full 250-sample window
(Meeting 23), with the RR features joined before the output layer; class weighting or resampling for
the rare classes; features normalised within each recording. With 22 development records, grouped
cross-validation is the only honest way to choose between them.